# Binary classification with random forest

## Re-process dataset into a simpler csv with an explicit 'source' column

this will later be used in the dataset. 

In [ ]:
from pathlib import Path
import pandas as pd
import yaml

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)

for v in Path(config["dataset"]["path"]).glob("*.dat"):
    print(v)
    source = None
    if "AGN" in v.name:
        source = 0
    elif "POPSTAR" in v.name:
        source = 1
    else:
        raise ValueError(f"Unknown source for {v.name}")

    df = pd.read_csv(
        v,
        sep=r"\s+",
        **(config["dataset"]["read_kwargs"] or {}),
        engine=config["dataset"]["engine"],
        comment=config["dataset"]["comment"],
    )

    df["source"] = source

    print(df.head())

    df.to_csv(v.with_suffix(".csv"), index=False, sep=",")

In [ ]:
from GalaxySpectrumClassifier import PandasDataset, to_xy
from torch.utils.data import random_split
import torch
import yaml

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset = PandasDataset.from_config(config["dataset"])
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

# Train model 

In [ ]:
from GalaxySpectrumClassifier import SimpleTrainer
import yaml

with open("../configs/binary_classsifier_simple_example.yaml", "r") as f:
    config = yaml.safe_load(f)
trainer = SimpleTrainer.from_config(config["trainer"])

trainer.fit(train_dataset)
trainer.save_snapshot("trained_random_forest")

# Test model

In [ ]:
from GalaxySpectrumClassifier import SimpleTrainer

trainer = SimpleTrainer.load_snapshot(
    "../training/binaryclassifier_simple_example/trained_random_forest"
)

In [ ]:
import pandas as pd
from GalaxySpectrumClassifier import SimpleTrainer

test_results = trainer.test(test_dataset)
test_results = pd.DataFrame.from_dict(
    data=[
        test_results,
    ]
)
test_results.to_csv(
    trainer.output_path / "trained_random_forest/test_results.csv", index=False
)
test_results

that these metrics are so pathologically high is an artifact of the data selection, not really of the quality of the classifier as such

## Random permutations cross-validation
this is used to get a handle on the influence of sample choice on the model. Here, because of the aforementioned reason, the results are always the same and only the losses are a little bit different. 

In [ ]:
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

X, y = to_xy(
    dataset, with_impute=False
)  # if impute=True, we end up with data leakage because the imputer is fitted on the entire dataset, including the test folds.

for train_idx, test_idx in kfold.split(
    X,
    y,
):
    train_dataset = torch.utils.data.Subset(dataset, train_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    trainer.fit(train_dataset)
    test_results = trainer.test(test_dataset)
    test_results = pd.DataFrame.from_dict(
        data=[
            test_results,
        ]
    )

    print(test_results)

# Binary classification with torch-based 2-layer multilayer perceptron

In [ ]:
from GalaxySpectrumClassifier import PandasDataset
from torch.utils.data import random_split
import torch
import yaml

with open("../configs/binary_classsifier_simple_example_torch.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset = PandasDataset.from_config(config["dataset"])
train_dataset, test_dataset = random_split(
    dataset, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

trainer = SimpleTrainer.from_config(config["trainer"])

trainer.fit(train_dataset)
trainer.save_snapshot("trained_mlp")

In [ ]:
import pandas as pd
from GalaxySpectrumClassifier import SimpleTrainer

trainer = SimpleTrainer.load_snapshot(
    "../training/binaryclassifier_simple_example_torch/trained_mlp"
)
test_results = trainer.test(test_dataset)
test_results = pd.DataFrame.from_dict(
    data=[
        test_results,
    ]
)
test_results.to_csv(trainer.output_path / "trained_mlp/test_results.csv", index=False)
test_results

## apply StratifiedKFold with the torch model 

In [ ]:
from sklearn.model_selection import StratifiedKFold
import yaml
import torch

with open("../configs/binary_classsifier_simple_example_torch.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset = PandasDataset.from_config(config["dataset"])
kfold = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

X, y = to_xy(
    dataset, with_impute=False
)  # if impute=True, we end up with data leakage because the imputer is fitted on the entire dataset, including the test folds.

for train_idx, test_idx in kfold.split(
    X,
    y,
):
    train_dataset = torch.utils.data.Subset(dataset, train_idx)
    test_dataset = torch.utils.data.Subset(dataset, test_idx)

    trainer.fit(train_dataset)
    test_results = trainer.test(test_dataset)
    test_results = pd.DataFrame.from_dict(
        data=[
            test_results,
        ]
    )

    print(test_results)